In [33]:
%pip install pandas scikit-learn nltk openpyxl

import os
import re
import string
import numpy as np
import pandas as pd
import nltk


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sklearn.preprocessing import OneHotEncoder
from sklearn.decomposition import TruncatedSVD  
from sklearn.preprocessing import StandardScaler 
from sklearn.pipeline import Pipeline  
from sklearn.linear_model import LogisticRegression 
from sklearn.model_selection import cross_val_score 

Note: you may need to restart the kernel to use updated packages.


In [34]:
# MBTI dataset (CSV)
mbti = pd.read_csv("myer-briggs-data.csv")
mbti.columns = ["type", "post"]
mbti


,type,post
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...
...,...,...
8670,ISFP,'https://www.youtube.com/watch?v=t8edHB_h908||...
8671,ENFP,'So...if this thread already exists someplace ...
8672,INTP,'So many questions when i do these things. I ...
8673,INFP,'I am very conflicted right now when it comes ...


In [35]:
#extroverted dataset
extroverted = pd.read_csv("dataset/extroverted.csv", header=None, engine="python")
extroverted.columns = ["type"] + ["post"] * (extroverted.shape[1] - 1)
extroverted = extroverted.replace(r"^[\s/]+$", "NaN", regex=True)

test_set = extroverted[extroverted["type"] == "<UNKNOWN>"].copy()
# remove those rows from the extroverted dataset
extroverted = extroverted[extroverted["type"] != "<UNKNOWN>"].copy()

extroverted = extroverted.dropna(axis=1, how='all')

extroverted


,type,post,post,post,post,post,post,post,post,post,...,post,post,post,post,post,post,post,post,post,post
0,ENTJ,"QQQ Mailbox CV: Don't get mad, you know at lea...",Tura,I'm taking you away. Up above the marshes,above the valleys,over the mountains and the forest,over the clouds and the sea,over the sun,over the clouds,over the limits of the world of stars,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ESTP,# To marry Zhong Han Liang #|||Zhong Han Liang...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ISFJ,"However, the wood belt / / @Green Bean Fries: ...",and I've always reported my phone number: 13**,***,****. Get used to it|||Thank you. It's like twit,too? Take care of yourself! Be safe! Be carefu...,Tsinjin Fire Command: The explosives are mainl...,mourn|||# Happy birthday to Fedler # # Happy b...,Fredler|||Happy birthday to Fedler|||[Poor] / ...,by the way|||Loandy knows best,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,INFJ,Ha|||Ha-ha-ha-ha-ha|||@RubyDedidi:Repost|||# h...,innocent boy's walls,hectic president's beverage,/ / star's red sugar,warm man's I. / / / / / / / / / / / / / / / / ...,but why not..|||||||||MY FRIEND IS A FOOL: HE ...,NOT ONLY IS HE SUPERIOR,BUT HE MUST BE PATIENT. YOU GUYS DID IT. I'LL ...,sad..|||It's so warm|||Aren't you my monk|||||...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ISTP,I don't know if there's any yogurt|||Once almo...,slow down|||#MILLION LOVE WEILAND # THE CONTIN...,and now it's the New Wave,and it's full fire,and it's going to be fun for you. You,you,you,you,you're all down! 26,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,ENFJ,|||[crying] keep your heart on the back of you...,and you'll get a little secret from me. @shard...,YOU DROPPED YOUR SOAP|||吖/[Turn] shared Slash ...,whose guitar is here... http://t.cn/8kwQexH|||...,the more you get to drink,the more you get to work,the more you get to walk by yourself... Is the...,"template free of charge # I'm using the ""Rolly...",wave face. Watch the video and learn more! htt...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
125,ENFJ,"|||Leave your footprints, say something. Let m...",the sky goes down,there's nothing going on. =|||||||||I don't wa...,the binary sails for the dream of a homemade a...,corrected by elderman == == sync,corrected by elderman == @elder_man|||It's a r...,and it's time for me to sleep. I'm not done wi...,but the problem has to be solved. I've been lo...,but there's more than an 8 level behind it. =|...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
126,ISFJ,"toming #beautiful life""|||[Haha]|||SOMETIMES I...",AND THE WORLD MUST BE WRONG|||#http://t.cn/RGO...,and I smiled at the screen,and I kept the girl away|||#bilibili# [Hinesh ...,but the twat buys them to satisfy [thinking] /...,but not for the love of you|||new year's wish:...,ta knows that it hurts when you practice it|||...,but I'm like a balloon,and I don't have the power to study anymore|||...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
127,ISTJ,@DINDING AND THE MINI-WOW CONCERTS WERE CALLED...,aah,"aah!"" This is my last chance! @StafCetaphil Xiaof",please [Cry tears] [Cry tears] [Cry tears] [Cr...,and I waited a year to see him [Cry tears] [Cr...,please [Cry tears] [Cry tears] [Cry tears] [Cr...,and I waited a year to see him [Cry tears] [Cr...,ah,ah! This is my last chance! @StafCetaphil Xiaof,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
# melt all post columns into one column named "post"
# rename ALL post columns to unique names: post_0, post_1, post_2, ...
post_cols = [col for col in extroverted.columns if col != "type"]
extroverted = extroverted.rename(
    columns={col: f"post_{i}" for i, col in enumerate(post_cols)}
)

# now melt safely
long_extroverted = extroverted.melt(
    id_vars=["type"],
    value_vars=[col for col in extroverted.columns if col.startswith("post_")],
    var_name="post_number",
    value_name="post"
)

# clean up
long_extroverted = long_extroverted.dropna(subset=["post"])
long_extroverted = long_extroverted[long_extroverted["post"] != ""]
long_extroverted = long_extroverted.reset_index(drop=True)
long_extroverted = long_extroverted.drop(columns=["post_number"])

long_extroverted

,type,post
0,ENTJ,"QQQ Mailbox CV: Don't get mad, you know at lea..."
1,ESTP,# To marry Zhong Han Liang #|||Zhong Han Liang...
2,ISFJ,"However, the wood belt / / @Green Bean Fries: ..."
3,INFJ,Ha|||Ha-ha-ha-ha-ha|||@RubyDedidi:Repost|||# h...
4,ISTP,I don't know if there's any yogurt|||Once almo...
...,...,...
81420,INFJ,SKYCATS ACCOUNT FOR 53.3 PER CENT
81421,INFJ,KYOTO|||In the middle of the night
81422,INFJ,who else would want to die of poverty in the v...
81423,INFJ,Ha ha! Is that a quick turn of mind or somethi...


In [37]:
mbti = pd.concat([mbti, long_extroverted], ignore_index=True)
mbti.to_csv("updated_mbti.csv", index=False)

In [6]:
mbti = pd.concat([mbti, long_extroverted], ignore_index=True)

In [6]:
mbti = pd.concat([mbti, long_extroverted], ignore_index=True)

In [7]:
mbti = mbti[mbti["post"].notna() & (mbti["post"].str.strip() != "") & (mbti["post"].str.strip() != "NaN")]
#nan_mbti = mbti["post"].isna().sum()
#nan_mbti

In [8]:
mbti.shape

(87084, 2)

In [9]:
###TASK 1: Dataset Inspection###
#inspecting the datasets
def inspect_dataset(df,name="Dataset"):
  print(f"/n====={name} INSPECTION=====")

#looking through shape and info about the datasets
  print(f"Shape: {df.shape}")
  print(f"Columns: {df.columns}")
  print("Info:")
  print(df.info())

  print("\nHead:")
  print(df.head())
#missing values
  print("\nMissing Values per Column:")
  print(df.isnull().sum())

#summary statistics
  print("\nSummary Statistics:(numeric columns):")
  print(df.describe())

# Inspecting all the datasets
inspect_dataset(mbti, "MBTI Dataset")

###TASK 2: Standardization Diagnostics:###
# Step 1: Align Column Names
def standardize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )
    return df

mbti = standardize_columns(mbti)

/n=====MBTI Dataset INSPECTION=====
Shape: (87084, 2)
Columns: Index(['type', 'post'], dtype='object')
Info:
<class 'pandas.core.frame.DataFrame'>
Index: 87084 entries, 0 to 90099
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   type    87084 non-null  object
 1   post    87084 non-null  object
dtypes: object(2)
memory usage: 2.0+ MB
None

Head:
   type                                               post
0  INFJ  'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1  ENTP  'I'm finding the lack of me in these posts ver...
2  INTP  'Good one  _____   https://www.youtube.com/wat...
3  INTJ  'Dear INTP,   I enjoyed our conversation the o...
4  ENTJ  'You're fired.|||That's another silly misconce...

Missing Values per Column:
type    0
post    0
dtype: int64

Summary Statistics:(numeric columns):
         type   post
count   87084  87084
unique     16  78228
top      INFJ     ha
freq    28179    710


In [11]:
### TASK 3: Clean data, remove stopwords and tokenize text ###


# Prepare stop words (combine sklearn's stop words with NLTK's if available)
stop_words = set(ENGLISH_STOP_WORDS)
try:
    from nltk.corpus import stopwords as nltk_stop
    stop_words = stop_words.union(set(nltk_stop.words('english')))
except Exception:
    # NLTK stopwords may not be downloaded in this environment; it's ok to proceed
    pass

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(tokens)

mbti['cleaned_text'] = mbti.get('post', '') .apply(clean_text) if 'post' in mbti.columns else ''
mbti["cleaned_text"] = mbti["post"].apply(clean_text)

#vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)
vectorizer = TfidfVectorizer(
    max_features=10000, 
    ngram_range=(1,3), 
    min_df=3
)
X = vectorizer.fit_transform(mbti["cleaned_text"])
if 'cleaned_text' in mbti.columns and mbti['cleaned_text'].astype(bool).any():
    X = vectorizer.fit_transform(mbti['cleaned_text'])
    y = mbti['type'] if 'type' in mbti.columns else None
else:
    X = None
    y = None

mbti.head()

,type,post,cleaned_text
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...,intj moments sportscenter plays prankswhat lif...
1,ENTP,'I'm finding the lack of me in these posts ver...,im finding lack posts alarmingsex boring posit...
2,INTP,'Good one _____ https://www.youtube.com/wat...,good course say know thats blessing cursedoes ...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o...",dear intp enjoyed conversation day esoteric ga...
4,ENTJ,'You're fired.|||That's another silly misconce...,youre firedthats silly misconception approachi...


In [19]:
### Task 2A-Prep: Encode MBTI Labels ###


# ---------- A) One-Hot Encoding for 16 MBTI types ----------
try:
    # sklearn >= 1.2 uses 'sparse_output'
    from sklearn.preprocessing import OneHotEncoder
    OHE_KW = dict(sparse_output=False, handle_unknown='ignore')

    _ = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
except TypeError:

    from sklearn.preprocessing import OneHotEncoder
    OHE_KW = dict(sparse=False, handle_unknown='ignore')

if 'type' not in mbti.columns:
    raise ValueError("Expected a 'type' column in the MBTI dataset.")

# Fit & transform
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe = encoder.fit_transform(mbti[['type']])

# Build DF with aligned index
ohe_df = pd.DataFrame(
    ohe,
    columns=encoder.get_feature_names_out(['type']),
    index=mbti.index
)

# Merge back
mbti_ohe = pd.concat([mbti, ohe_df], axis=1)

# B) Binary Axes (I/E, N/S, T/F, J/P)
def split_axes(mbti_type: str):
    t = str(mbti_type).upper()
    if len(t) != 4:
        return pd.Series([np.nan, np.nan, np.nan, np.nan], index=["IE","NS","TF","JP"])
    return pd.Series([t[0], t[1], t[2], t[3]], index=["IE","NS","TF","JP"])

axes = mbti_ohe['type'].apply(split_axes)

label_maps = {
    "IE": {"I": 1, "E": 0},
    "NS": {"N": 1, "S": 0},
    "TF": {"T": 1, "F": 0},
    "JP": {"J": 1, "P": 0},
}

for col, mapping in label_maps.items():
    axes[col + "_y"] = axes[col].map(mapping)

# Combine with the MBTI one-hot dataset
mbti_encoded = pd.concat([mbti_ohe, axes], axis=1)

In [22]:
clf_16 = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)),
    ("svd", TruncatedSVD(n_components=200)),
    ("logreg", LogisticRegression(max_iter=3000))
])

scores_16 = cross_val_score(clf_16, mbti["cleaned_text"], mbti["type"], cv=5)
print("16-class accuracy:", scores_16.mean())

axes_cols = ["IE_y", "NS_y", "TF_y", "JP_y"]

for col in axes_cols:
    clf = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)),
        ("svd", TruncatedSVD(n_components=200)),
        ("logreg", LogisticRegression(max_iter=3000)),
    ])

    y = mbti_encoded[col].squeeze().astype(int)  # ensures 1D and integer type
    # ensure it's a 1D array
    scores = cross_val_score(
        clf,
        mbti_encoded["cleaned_text"],
        y,
        cv=5
    )
    
    print(f"{col} accuracy: {scores.mean():.4f}")

16-class accuracy: 0.34199164346873173


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/pipeline.py", line 662, in fit
    self._final_estimator.fit(Xt, y, **last_step_params["fit"])
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py

In [62]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1,2),
        min_df=3
    )),
    ("svm", LinearSVC(class_weight="balanced"))
])

from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

clf = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1,2),
        min_df=3
    )),
    ("svm", LinearSVC(class_weight="balanced"))
])

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    mbti["cleaned_text"], 
    mbti["type"],
    test_size=0.2,
    random_state=42,
    stratify=mbti["type"]
)

clf.fit(X_train, y_train)

from sklearn.metrics import accuracy_score

y_pred = clf.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))


Test accuracy: 0.34868232186943793
